##### Copyright 2026 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Robotics ER 2 クイックスタート（日本語版）

IoP技術者コミュニティ R8年度 第4回講座（フィジカルAI編）向けに、Google 公式ノートブック
[gemini_robotics_er.ipynb](https://github.com/google-gemini/robotics-samples/blob/main/Getting%20Started/gemini_robotics_er.ipynb)
の説明文を日本語化したものです（コードは公式のまま。Apache-2.0）。

このノートブックでは **Gemini Robotics ER 2** モデルを紹介します。

Gemini Robotics ER 2 は、ロボットが物理世界を知覚し、働きかけるための視覚言語モデル（VLM）です。
視覚データを解釈し、空間・時間の推論を行い、複数ステップのタスクを計画し、ロボットやツールを協調させます。

扱うモデルは `gemini-robotics-er-2-preview` です。Gemini 3.5 Flash をベースに、空間推論・動画のモーメント検出・
動画の進捗分類・複数ロボットの協調・複数ステップのツール使用が強化されています。

セクションごとに、次の能力を順に見ていきます。
- **空間推論**: 物を指す、動画内で追跡する、バウンディングボックスで検出する、軌跡を計画する
- **エージェント的なコード実行**: コード実行で画像を拡大・切り出しし、計器を読み、シーンに注釈を付ける
- **タスクの組み立て**: 空間推論と自作のロボットAPIを組み合わせ、長いタスクを完了させる
- **動画の進捗理解**: 連続した動画からのモーメント検出と進捗分類


## 1. セットアップ

このセクションは、Colab を起動するたびに実行してください。SDK を導入し、API クライアントを初期化します。

**先に、左の鍵アイコン（シークレット）に `GEMINI_API_KEY` を登録し、「ノートブックからのアクセス」を ON にしておいてください。**


In [ ]:
%pip install -U -q "google-genai>=2.9.0" pydantic japanize-matplotlib

In [ ]:
from google.colab import userdata
from google import genai

try:
  GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
  print("ERROR: Please add your GEMINI_API_KEY in the Secrets section (key icon 🔑 in the left sidebar) and then enable access to it.")

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_ID = "gemini-robotics-er-2-preview"
response = client.interactions.create(model=MODEL_ID, input="Hello Physical World?")
print(response.output_text)

### ユーティリティと可視化ヘルパー


In [ ]:
# 標準ライブラリ
import asyncio
import base64
import concurrent.futures
import json
import time
from io import BytesIO
from typing import List

# 外部ライブラリ
import cv2
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib  # 日本語ラベルの文字化け防止（講座で追加）
import matplotlib.patches as patches
from PIL import Image
from pydantic import BaseModel, Field
from IPython.display import display, Image as IPythonImage

In [ ]:
def load_image(image_path):
    """画像を読み込み、トークン節約のため縮小して、PIL 画像と base64 文字列の両方を返す。"""
    img = Image.open(image_path)
    img = img.resize((800, int(800 * img.size[1] / img.size[0])), Image.Resampling.LANCZOS)

    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode()
    return img, img_b64

def plot_points(img, json_data):
    """返ってきた点（Pydantic 形式）を Matplotlib で画像に描く。"""
    w, h = img.size

    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')

    try:
        if isinstance(json_data, list):
            data = json_data
        else:
            data = json.loads(json_data.strip()).get("items", [])
    except Exception:
        data = []

    for item in data:
        if "point" not in item: continue
        y, x = item["point"]
        px, py = (x / 1000) * w, (y / 1000) * h

        plt.plot(px, py, 'wo', markeredgecolor='#0057FF', markersize=8, markeredgewidth=2)
        plt.plot([px, px + 25], [py, py - 20], color='#0057FF', linewidth=2)
        plt.text(px + 25, py - 20, item.get("label", ""), color='white',
                 bbox=dict(boxstyle="round,pad=0.3", facecolor='#0057FF', edgecolor='none'))

    # plt.show() の代わりに JPEG 圧縮で保存して、ノートブックを軽くする
    buf = BytesIO()
    plt.savefig(
        buf,
        format='jpg',
        bbox_inches='tight',
        pad_inches=0,
        pil_kwargs={'quality': 60, 'optimize': True}
    )
    plt.close()

    display(IPythonImage(data=buf.getvalue()))

def plot_bounding_boxes(img, json_data):
    """返ってきたバウンディングボックス（Pydantic 形式）を Matplotlib で画像に描く。"""
    w, h = img.size

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img)
    ax.axis('off')

    try:
        data = json.loads(json_data.strip()).get("items", [])
    except Exception:
        data = []

    for item in data:
        ymin, xmin, ymax, xmax = item.get("box_2d", item.get("box", [0,0,0,0]))
        px, py = (xmin / 1000) * w, (ymin / 1000) * h
        bw, bh = ((xmax - xmin) / 1000) * w, ((ymax - ymin) / 1000) * h

        ax.add_patch(patches.Rectangle((px, py), bw, bh, linewidth=2, edgecolor='#0057FF', facecolor='none'))
        ax.text(px, py - 5, item.get("label", ""), color='white',
                bbox=dict(boxstyle="round,pad=0.3", facecolor='#0057FF', edgecolor='none'))

    # plt.show() の代わりに JPEG 圧縮で保存して、ノートブックを軽くする
    buf = BytesIO()
    plt.savefig(
        buf,
        format='jpg',
        bbox_inches='tight',
        pad_inches=0,
        pil_kwargs={'quality': 60, 'optimize': True}
    )
    plt.close()

    display(IPythonImage(data=buf.getvalue()))

def display_compressed(ipy_image, quality=60):
    """
    IPython の Image を JPEG に圧縮して表示し、ノートブックの容量を節約する。
    """
    # 1. IPython Image の .data から生バイト列を読む
    img = Image.open(BytesIO(ipy_image.data))

    # 2. RGB に変換（元が RGBA/PNG の場合に必要）
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")

    # 3. 圧縮して新しいバッファに保存
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=quality, optimize=True)

    # 4. 圧縮後の画像を表示
    display(IPythonImage(data=buf.getvalue()))

## 2. 空間推論

まず、空間ロジック・バウンディングボックス・物体検出で使う画像をダウンロードします。


In [ ]:
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/aloha-arms-table.png -O aloha-arms-table.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/gameboard.png -O gameboard.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/washers.png -O washers.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/bookshelf.jpeg -O bookshelf.jpeg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/particles.jpg -O particles.jpg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/livingroom.jpeg -O livingroom.jpeg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/clear_space.png -O clear_space.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/lunch.png -O lunch.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/sockets.jpeg -O sockets.jpeg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/weights.jpeg -O weights.jpeg -q

### 2D ポインティング — 物の名前を指定せずに指す

「写っている物を全部指して」と聞きます。座標は `[y, x]`（**y が先**）、左上 0・右下 1000 の比率で返ります。


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('aloha-arms-table.png')
prompt = "写っている物を全部指してください。ラベルは、それぞれの物の名前を日本語で返してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "low"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 名前を指定して指す


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('aloha-arms-table.png')
queries = ["bread", "starfruit", "banana"]
prompt = f"次の物すべてに対応する点を返してください: {', '.join(queries)}。ラベルは物の名前を日本語で返してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 抽象的な説明に当てはまる物を全部指す


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('aloha-arms-table.png')
prompt = "果物を全部指してください。ラベルは実際の物の名前を日本語で返してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 同じ種類の物を全部指す


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('gameboard.png')
prompt = "ゲーム盤のマス（スロット）に対応する点を全部返してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 物の特定の部位をまとめて指す


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('aloha-arms-table.png')
points_data = []

queries = [
    ("paper bag", "handles"), ("banana", "the stem"), ("banana", "center"),
    ("starfruit", "center"), ("lime", "center"), ("light blue bowl", "rim"),
    ("measuring cup", "handle"), ("bowl", "tomato")
]

def process_query(obj, part):
    prompt = f"画像の中の {obj} の {part} を指してください。点は1つだけ返してください。"
    resp = client.interactions.create(
        model=MODEL_ID,
        input=[{"type": "user_input", "content": [
            {"type": "image", "data": img_b64, "mime_type": "image/png"},
            {"type": "text", "text": prompt}
        ]}],
        generation_config={"thinking_level": "low"},
        response_format={
            "type": "text", "mime_type": "application/json",
            "schema": PointResponse.model_json_schema()
        }
    )
    try:
        return json.loads(resp.output_text.strip()).get("items", [])
    except:
        return []

# 空間の質問は互いに独立なので、並列に投げると全体の待ち時間を大きく減らせる
with concurrent.futures.ThreadPoolExecutor() as executor:
    results = executor.map(lambda q: process_query(q[0], q[1]), queries)

for res in results:
    points_data.extend(res)

plot_points(img, json.dumps({"items": points_data}))

### 指しながら数える


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('washers.png')
prompt = "箱の中のワッシャーを1つずつ指してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

data = json.loads(response.output_text.strip()).get("items", [])
print(f"Count: {len(data)}")
plot_points(img, response.output_text)

### 2D バウンディングボックス — 四角で囲む


In [ ]:
class BBoxItem(BaseModel):
    box_2d: List[int] = Field(description="[ymin, xmin, ymax, xmax] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")
class BBoxResponse(BaseModel):
    items: List[BBoxItem]

img, img_b64 = load_image('aloha-arms-table.png')
prompt = "ラベル付きのバウンディングボックスを返してください。最大25個。机の上で識別できる物をできるだけ多く含めてください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "low"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": BBoxResponse.model_json_schema()
    }
)

plot_bounding_boxes(img, response.output_text)

### 空間ロジック — 特定の棚を見つける


In [ ]:
class BBoxItem(BaseModel):
    box_2d: List[int] = Field(description="[ymin, xmin, ymax, xmax] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class BBoxResponse(BaseModel):
    items: List[BBoxItem]

img, img_b64 = load_image('bookshelf.jpeg')
prompt = '"I need to blow my nose." Find the cubby that can help.'

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "low"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": BBoxResponse.model_json_schema()
    }
)

plot_bounding_boxes(img, response.output_text)

### 軌跡 — 経路を計画する


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('particles.jpg')
prompt = "青いブラシを指し、あわせて粒が散らばっている領域を覆う10個の点を返してください。点は粒の上に均等に配置し、なめらかな軌跡になるようにしてください。ラベルは通るべき順に 1〜10 の番号を付けてください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "low"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 障害物を避ける軌跡の計画


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('livingroom.jpeg')
prompt = "現在の視点の原点から左奥の緑のオットマンまで、床の上を通る衝突のない最短の軌跡を10個の点で示してください。床の上の他の障害物はすべて避けてください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "low"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 空間の関係 — 置き場所を空ける


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('clear_space.png')
prompt = "ノートPCを置く場所を空けるために、どかすべき物を指してください。"

# 複雑な論理推論のため thinking を high にする
response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 組み立て — お弁当を詰める


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointAndAnswerResponse(BaseModel):
    items: List[PointItem]
    answer: str

img, img_b64 = load_image('lunch.png')
prompt = "お弁当箱とランチバッグへの詰め方を説明してください。説明で触れた物は、それぞれ指してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointAndAnswerResponse.model_json_schema()
    }
)
response_dict = json.loads(response.output_text.strip())
plot_points(img, response_dict.get("items", []))
print("\n" + response_dict.get("answer", ""))

### 空いている電源ソケットを見つける


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('sockets.jpeg')
prompt = "ふさがっていない空きソケットを指してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 持ち上げ重量の制限（3ポンドまで）


In [ ]:
class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] 形式、0-1000 に正規化")
    label: str = Field(description="検出した物の名前（日本語で）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image('weights.jpeg')
prompt = "私は可搬重量 3ポンド（約1.4kg）のロボットです。画像の中で、物理的に持ち上げられる物をすべて指してください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema()
    }
)

plot_points(img, response.output_text)

### 複数視点の対応付けと成功判定

モデルは複数の視点の画像をまとめて推論し、物理的な状態の変化からタスクの成否を判定できます。

この例では、ロボットのカメラ映像から「マンゴーを茶色い容器に入れる」というタスクが完了したかを判定します。


In [ ]:
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/initial_state_1.png -O initial_state_1.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/initial_state_2.png -O initial_state_2.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/initial_state_3.png -O initial_state_3.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/initial_state_4.png -O initial_state_4.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/current_state_1.png -O current_state_1.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/current_state_2.png -O current_state_2.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/current_state_3.png -O current_state_3.png -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/current_state_4.png -O current_state_4.png -q

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))
fig.suptitle('Multi-view: Initial State vs Current State', fontsize=16)

for i in range(4):
    # 上段: 開始時点の状態
    axes[0, i].imshow(Image.open(f"initial_state_{i+1}.png"))
    axes[0, i].set_title(f"Initial State {i+1}")
    axes[0, i].axis('off')

    # 下段: 現在の状態
    axes[1, i].imshow(Image.open(f"current_state_{i+1}.png"))
    axes[1, i].set_title(f"Current State {i+1}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class SuccessResponse(BaseModel):
    success: bool = Field(description="タスクに成功していれば True、そうでなければ False")

prompt = (
    "このタスクでは、ロボット（または人）が「マンゴーを茶色い容器に入れる」作業をしている様子を見ます。"
    "同じシーンを複数のカメラ視点で見ることがあります。\n\n"
    "最初の4枚は、エピソード開始時点（少し前）の複数カメラの画像です。"
    "最後の4枚は、エピソードの現時点（今）の複数カメラの画像です。\n\n"
    "開始時点と現在を見比べて、ロボットはタスクを成功させましたか？"
)

content_list = []
for state in ['initial_state_1.png', 'initial_state_2.png', 'initial_state_3.png', 'initial_state_4.png',
              'current_state_1.png', 'current_state_2.png', 'current_state_3.png', 'current_state_4.png']:
    _, b64 = load_image(state)
    content_list.append({"type": "image", "data": b64, "mime_type": "image/png"})

content_list.append({"type": "text", "text": prompt})

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": content_list}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": SuccessResponse.model_json_schema()
    }
)

result = json.loads(response.output_text)
print(f"Success? {'Yes' if result.get('success') else 'No'}")

## 3. エージェント的な視覚（Agentic Vision）

このモデルは、回答の途中で **自分で Python コードを実行** できます。論理を確かめる、画像を拡大・切り出しする、
画像に注釈を描き込む、といったことを自分でやってから答えます。


In [ ]:
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/gemini-robotics-er/clock.jpg -O clock.jpg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/gemini-robotics-er/meter.jpeg -O meter.jpeg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/gemini-robotics-er/circuit_board.jpeg -O circuit_board.jpeg -q
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/gemini-robotics-er/sorting.jpeg -O sorting.jpeg -q

### 時計を読んで、論理を適用する


次の画像を使います。

![Clock](https://storage.googleapis.com/generativeai-downloads/images/robotics/gemini-robotics-er/clock.jpg)


In [ ]:
img, img_b64 = load_image('clock.jpg')

class ClockResponse(BaseModel):
    valid: bool
    hours: int
    minutes: int
    date: str | bool
    month: str | bool
    weekday: str | bool

prompt_time = "何時ですか？ 時計としてありえる正しい時刻ですか？ 回答には必ず、妥当性・時・分を含めてください。日付・月・曜日が明示されていれば、それも含めてください。"

response_time = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt_time}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": ClockResponse.model_json_schema()
    }
)
print("Time Response:\n", json.dumps(json.loads(response_time.output_text), indent=2))

### コード実行で計器を読む


In [ ]:
img, img_b64 = load_image('meter.jpeg')

prompt = (
    "液面計はどのくらい満たされていますか？\n"
    "読み方:\n"
    "1) のぞき窓の上端・下端・液面の3点を見つける\n"
    "2) 計算で液面の割合（%）を求める\n"
    '3) 最後に "Answer: ??" を単独の行で出力する（?? は % や単位を付けない数値）'
)

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    tools=[{"type": "code_execution"}],
)

for step in response.steps:
    if step.type == "code_execution_call":
        print(f"--- EXECUTING CODE ---\n{step.arguments.code}")
    elif step.type == "code_execution_result":
        print(f"--- EXECUTION RESULT ---\n{step.result}")
    elif step.type == "model_output":
        for content in step.content:
            if content.type == "text" and content.text:
                print(content.text)
            elif content.type == "image" and content.data:
                display_compressed(IPythonImage(data=base64.b64decode(content.data)))

### 拡大・切り出しで細かい文字を読む


次の基板の左上にある ESMT チップの型番を読み取らせます。
![Circuit Board](https://storage.googleapis.com/generativeai-downloads/images/robotics/gemini-robotics-er/circuit_board.jpeg)


In [ ]:
img, img_b64 = load_image('circuit_board.jpeg')

prompt = "ESMT チップの型番は何ですか？ Python で拡大・切り出し・回転して読み取ってください。"

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    tools=[{"type": "code_execution"}],
)

for step in response.steps:
    if step.type == "code_execution_call":
        print(f"--- EXECUTING CODE ---\n{step.arguments.code}")
    elif step.type == "code_execution_result":
        print(f"--- EXECUTION RESULT ---\n{step.result}")
    elif step.type == "model_output":
        for content in step.content:
            if content.type == "text" and content.text:
                print(content.text)
            elif content.type == "image" and content.data:
                display_compressed(IPythonImage(data=base64.b64decode(content.data)))

### 画像への注釈（仕分け）


In [ ]:
img, img_b64 = load_image('sorting.jpeg')

prompt = """
この画像を見て、どのゴミをどの分別容器に入れるべきかを、色の違う矢印で描き込んだ
注釈付き画像として返してください。最終的な画像を必ず API 呼び出し元に返してください。
指す場合は [y, x] 形式（0-1000 に正規化）を使ってください。
"""

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    tools=[{"type": "code_execution"}],
)

for step in response.steps:
    if step.type == "code_execution_call":
        print(f"--- EXECUTING CODE ---\n{step.arguments.code}")
    elif step.type == "code_execution_result":
        print(f"--- EXECUTION RESULT ---\n{step.result}")
    elif step.type == "model_output":
        for content in step.content:
            if content.type == "image" and content.data:
                display_compressed(IPythonImage(data=base64.b64decode(content.data)))

### エージェント的視覚での物体検出（仕分け）


In [ ]:
img, img_b64 = load_image('sorting.jpeg')

prompt = """
このシーンの中で堆肥化できる（コンポスト行きの）物について、[ymin, xmin, ymax, xmax]
形式（0-1000 に正規化）のバウンディングボックスを返してください。よく見えるように画像を
拡大・切り出ししてください。処理の一部として、最終結果のボックスを描き込んだ注釈付き画像も
API 呼び出し元に返してください。
"""

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    generation_config={"thinking_level": "high"},
    tools=[{"type": "code_execution"}],
)

for step in response.steps:
    if step.type == "code_execution_call":
        print(f"--- EXECUTING CODE ---\n{step.arguments.code}")
    elif step.type == "code_execution_result":
        print(f"--- EXECUTION RESULT ---\n{step.result}")
    elif step.type == "model_output":
        for content in step.content:
            if content.type == "text" and content.text:
                print(content.text)
            elif content.type == "image" and content.data:
                display_compressed(IPythonImage(data=base64.b64decode(content.data)))

## 4. タスクの組み立て

自作のロボット API と組み合わせたタスクの組み立てを示すために、シンプルなつかんで置く操作（pick-and-place）用の
模擬 API を使います。


次の画像を分析します。

![SO101](https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/soarm-block.png)


In [ ]:
!wget https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/soarm-block.png -O soarm-block.png -q

img, img_b64 = load_image('soarm-block.png')

# 模擬ロボット API の定義
def move(x, y, high):
    print(f"Mock Robot: Moving to coordinates: {x}, {y}, {'high above table' if high else 'down at table level'}")

def setGripperState(opened):
    print(f"Mock Robot: {'Opening gripper' if opened else 'Closing gripper'}")

robot_origin_y = 300
robot_origin_x = 500

move_function = {
    "type": "function",
    "name": "move",
    "description": "アームを指定した座標へ動かす。",
    "parameters": {
        "type": "object",
        "properties": {
            "x": {"type": "integer", "description": "原点からの相対 X 座標"},
            "y": {"type": "integer", "description": "原点からの相対 Y 座標"},
            "high": {"type": "boolean", "description": "True でアームを高く上げて障害物を避ける。False でグリッパーを作業面まで下ろす。"}
        },
        "required": ["x", "y", "high"]
    }
}

set_gripper_state_function = {
    "type": "function",
    "name": "setGripperState",
    "description": "グリッパーを開閉する。",
    "parameters": {
        "type": "object",
        "properties": {
            "opened": {"type": "boolean", "description": "True でグリッパーを開き、False で閉じる。"}
        },
        "required": ["opened"]
    }
}

prompt = (
    "あなたは6自由度のロボットアームです。"
    f"移動計算の原点は、正規化座標 y={robot_origin_y}, x={robot_origin_x} にあります。"
    "この点を新しい (0,0) として移動を計算してください（x, y は負の値も可）。\n\n"
    "青いブロックとオレンジのボウルを見つけ、原点からの相対座標を計算してください。\n"
    "青いブロックをつかんでオレンジのボウルに入れる、つかんで置く操作を実行してください。"
    "この操作を完了するために、適切な順序で関数を呼び出してください。"
)

# 1. 最初の呼び出し
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt}
    ]}],
    tools=[move_function, set_gripper_state_function],
    generation_config={"thinking_level": "low"}
)

print("\n--- Executing Orchestrated Plan ---")

max_steps = 15 # Safety limit to prevent infinite loops, best practice to prevent infinite tool-calling loops.
step_count = 0

# 2. エージェントのループ
while step_count < max_steps:
    step_count += 1

    # モデルが関数を呼ぼうとしているか確認
    tool_calls = [step for step in interaction.steps if step.type == "function_call"]

    if not tool_calls:
        # 関数呼び出しが無ければ、一連の操作は完了
        print("Sequence complete.")
        if interaction.output_text:
            print(f"Model Summary: {interaction.output_text}")
        break

    function_results = []

    for step in tool_calls:
        function_name = step.name
        arguments = step.arguments

        # 模擬関数を実行
        if function_name == "move":
            move(**arguments)
        elif function_name == "setGripperState":
            setGripperState(**arguments)
        else:
            print(f"Unknown function: {function_name}")

        # 3. 関数が成功したことをモデルに伝える結果オブジェクトを作る
        function_results.append({
            "type": "function_result",
            "name": step.name,
            "call_id": step.id,
            "result": [{"type": "text", "text": '{"status": "success"}'}]
        })

    # 4. 結果をモデルに返す。previous_interaction_id を渡すことで
    #    会話の履歴を覚えたまま、次のステップを生成させる
    interaction = client.interactions.create(
        model=MODEL_ID,
        previous_interaction_id=interaction.id,
        tools=[move_function, set_gripper_state_function],
        input=function_results
    )

## 5. 動画の進捗理解

Gemini Robotics ER 2 は連続した動画の分析が得意です。進捗の把握、特定の瞬間の検出、意味的な評価ができます。
動画は API に直接アップロードし、フレーム処理はモデル側に任せます。


### 動画分析と状態の保持

Interactions API では、会話の状態がサーバー側に自動保存されます。動画を一度アップロードすれば、
全体の分析を頼んだあと、再アップロードなしでタイムスタンプを指定した追加質問ができます。


次の動画を分析します。

[https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/desk_organization.mp4](https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/desk_organization.mp4)
<video controls width="100%" src="https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/desk_organization.mp4">
</video>


In [ ]:
!wget -O desk_organization.mp4 "https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/desk_organization.mp4" -q

In [ ]:
print("Uploading desk_organization.mp4...")
video_file = client.files.upload(file="desk_organization.mp4")

while video_file.state.name == "PROCESSING":
    print(".", end="", flush=True)
    time.sleep(2)
    video_file = client.files.get(name=video_file.name)

if video_file.state.name == "FAILED":
    raise ValueError(f"File processing failed: {video_file.uri}")

print(f"\nReady: {video_file.uri}")

In [ ]:
class VideoEvent(BaseModel):
    start_timestamp: str
    end_timestamp: str
    description: str

class VideoAnalysis(BaseModel):
    events: List[VideoEvent]

prompt_1 = "タスクを終えるまでの各ステップを詳しく説明してください。タイムスタンプで区切ってください。"

# 1回目の質問
response_1 = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": video_file.uri},
        {"type": "text", "text": prompt_1}
    ]}],
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": VideoAnalysis.model_json_schema()
    }
)
print("\n--- Initial Broad Analysis ---")
print(json.dumps(json.loads(response_1.output_text), indent=2))

In [ ]:
prompt_2 = "15秒から22秒の区間を詳しく見て、1秒ごとに何が起きているかを説明してください。"

response_2 = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "text", "text": prompt_2}
    ]}],
    previous_interaction_id=response_1.id,
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": VideoAnalysis.model_json_schema()
    }
)
print("\n--- Granular Follow-up Analysis ---")
print(json.dumps(json.loads(response_2.output_text), indent=2))

### モーメント検出

動画の中で、ある出来事が起きた正確な時刻を見つけます。

ここでは「持っているドライバーを置く」というタスクが完了した時刻を特定します。

[https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/moment_finding.mp4](https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/moment_finding.mp4)
<video controls width="100%" src="https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/moment_finding.mp4">
</video>


In [ ]:
!wget -O moment_finding.mp4 "https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/moment_finding.mp4" -q

print("Uploading moment_finding.mp4...")
moment_video = client.files.upload(file="moment_finding.mp4")

while moment_video.state.name == "PROCESSING":
    print(".", end="", flush=True)
    time.sleep(2)
    moment_video = client.files.get(name=moment_video.name)

print(f"\nReady: {moment_video.uri}")

class MomentResponse(BaseModel):
    final_answer: str = Field(description="タイムスタンプ（例: 00:06）。見つからなければ -1")

MOMENT_FINDING_PROMPT = """
ロボット（または人）がタスクを行う様子を観察します。各フレームは複数カメラのグリッドで、
左から右・上から下に cage_back_rgb_img, cage_left_rgb_img, cage_right_rgb_img,
wrist_back_left_rgb_img, wrist_back_right_rgb_img の順に並んでいます。
タスクは「持っているドライバーを、机の上のドライバーの右側に置く」です。
ロボット（または人）がタスクを成功させた正確なタイムスタンプを特定してください。
"""

response_moment = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": moment_video.uri},
        {"type": "text", "text": MOMENT_FINDING_PROMPT}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": MomentResponse.model_json_schema()
    }
)

print("\nMoment Finding Output:")
print(json.dumps(json.loads(response_moment.output_text), indent=2))

#### 可視化ヘルパー


In [ ]:
def flatten_views(frame):
    """2x3 グリッドのフレームを横一列（1x5）に並べ替える。"""
    h, w, _ = frame.shape
    vh, vw = h // 2, w // 3

    # 5つのカメラ視点を切り出す
    v1 = frame[0:vh, 0:vw]
    v2 = frame[0:vh, vw:2*vw]
    v3 = frame[0:vh, 2*vw:w]
    v4 = frame[vh:h, 0:vw]
    v5 = frame[vh:h, vw:2*vw]

    # 横一列に連結する
    return np.concatenate([v1, v2, v3, v4, v5], axis=1)

def plot_video_moment(video_path, json_data, num_frames=10):
    # 構造化 JSON からタイムスタンプを取り出す
    data = json.loads(json_data.strip())
    timestamp_str = data.get("final_answer", "-1")

    if timestamp_str == "-1":
        print("Task was not completed in the video.")
        return

    # 「MM:SS」または「SS」を秒に変換
    parts = timestamp_str.split(":")
    target_sec = int(parts[0]) * 60 + int(parts[1]) if len(parts) == 2 else int(timestamp_str)

    # 動画を開いてプロパティを取得
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps if fps > 0 else 10

    # 動画全体から等間隔に時刻をサンプリング
    times = np.linspace(0, duration, num_frames, endpoint=False)

    # 対象のタイムスタンプが必ずフレームに含まれるようにする
    target_idx = np.argmin(np.abs(times - target_sec))
    times[target_idx] = target_sec

    # フレームを切り出して並べ替える
    frames = []
    for t in times:
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(flatten_views(frame_rgb))
        else:
            frames.append(np.zeros((100, 500, 3), dtype=np.uint8)) # Fallback
    cap.release()

    # フレームを縦に積んで、余白や隙間をゼロにする
    full_image = np.vstack(frames)
    H, W, _ = frames[0].shape

    # 幅を1/4にしてアスペクト比を保って描画
    fig_height = 4 * (full_image.shape[0] / full_image.shape[1])
    fig, ax = plt.subplots(figsize=(8, fig_height))

    # 図の枠と余白をすべて除去
    fig.subplots_adjust(top=1, bottom=0, right=1, left=0, hspace=0, wspace=0)
    ax.margins(0, 0)
    ax.axis('off')

    ax.imshow(full_image)

    # モデルが返した瞬間のフレームを強調表示
    y_offset = target_idx * H
    lw = 8 # Border thickness
    rect = patches.Rectangle((lw/2, y_offset + lw/2), W - lw, H - lw,
                             linewidth=lw, edgecolor='#0057FF', facecolor='none')
    ax.add_patch(rect)

    # 右下にタイムスタンプのバッジ
    ax.text(W, y_offset + H, f" Timestamp: {timestamp_str} ", color='white',
            fontsize=16, fontweight='bold', ha='right', va='bottom',
            bbox=dict(boxstyle="square,pad=0.4", facecolor='#0057FF', edgecolor='none'))

    # Colab 上で見やすいように画像を圧縮
    buf = BytesIO()
    plt.savefig(
        buf,
        format='jpg',
        bbox_inches='tight',
        pad_inches=0,
        pil_kwargs={'quality': 60, 'optimize': True}
    )
    plt.close()

    display(IPythonImage(data=buf.getvalue()))

#### 可視化


In [ ]:
plot_video_moment("moment_finding.mp4", response_moment.output_text)

### 進捗分類

動画内でどれだけ作業が進んだかを、意味的なカテゴリ（1〜4）で返します。

この例では、次の動画でどこまで進んだかを判定します。

[https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/progress_classification_04.mp4](https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/progress_classification_04.mp4)
<video controls width="100%" src="https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/progress_classification_04.mp4">
</video>


#### 可視化ヘルパー


In [ ]:
def plot_video_progress(video_path, json_data, label_prefix="Level", num_frames=5):
    # 1. 結果をパース
    data = json.loads(json_data.strip())
    choice = data.get("digit_choice", "?")

    # 2. フレームを切り出して並べ替える（2x2 → 1x4）
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frames = []
    # タイムライン表示用に等間隔でフレームをサンプリング
    for f_idx in np.linspace(0, total_frames-1, num_frames, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
        ret, f = cap.read()
        if not ret: continue
        h, w = f.shape[:2]

        # 2x2 グリッドを 1x4 の横並びにする
        frames.append(np.hstack([f[:h//2, :], f[h//2:, :]]))
    cap.release()

    # 3. 縦に積んで描画
    img = cv2.cvtColor(np.vstack(frames), cv2.COLOR_BGR2RGB)
    Total_H, W = img.shape[:2] # Get dimensions of the FULL stacked image

    fig, ax = plt.subplots(figsize=(8, 8 * Total_H / W))
    fig.subplots_adjust(0,0,1,1,0,0); ax.axis('off')
    ax.imshow(img)

    # 4. 全フレームをひとつの枠で囲んで強調
    lw = 8
    ax.add_patch(plt.Rectangle((lw/2, lw/2), W - lw, Total_H - lw,
                               linewidth=lw, edgecolor='#0057FF', fill=False))

    # 右下に判定結果のバッジ
    ax.text(W, Total_H, f" {label_prefix}: {choice} ", color='white',
            fontsize=20, fontweight='bold', ha='right', va='bottom',
            bbox=dict(color='#0057FF', pad=0.4))

    # GitHub 上で見やすいように画像を圧縮
    buf = BytesIO()
    plt.savefig(
        buf,
        format='jpg',
        bbox_inches='tight',
        pad_inches=0,
        pil_kwargs={'quality': 60, 'optimize': True}
    )
    plt.close()

    display(IPythonImage(data=buf.getvalue()))

#### 進捗分類（パーセント）


In [ ]:
!wget -O progress.mp4 "https://storage.googleapis.com/generativeai-downloads/images/robotics/er-example-colab/progress_classification_04.mp4" -q

print("Uploading progress.mp4...")
progress_video = client.files.upload(file="progress.mp4")

while progress_video.state.name == "PROCESSING":
    print(".", end="", flush=True)
    time.sleep(2)
    progress_video = client.files.get(name=progress_video.name)

print(f"\nReady: {progress_video.uri}")

In [ ]:
class PercentProgressResponse(BaseModel):
    digit_choice: int = Field(description="1〜5 の整数の選択肢")
    reasoning: str

PERCENTAGE_PROGRESS_PROMPT = (
    "In the following task, you will see a robot or human trying to perform a task to "
    "insert a toy screw into the top left hole on the workbench.\n\n"
    "You will see multiple camera views of the same scene. Some cameras are static and are "
    "mounted outside of the scene and some cameras are mounted on the robot arms and thus "
    "they are moving during the episode.\n\n"
    "Each image shows 4 camera views concatenated together.\n\n"
    "Considering the progress shown in the video wherein the last image in the video sequence "
    "shows the current state of the task, what is the progress percentage at the end of the "
    "episode towards the task \"insert the toy screw into the top left hole on the workbench\"?\n\n"
    "Answer with choices:\n"
    "1) 0-20\n"
    "2) 21-40\n"
    "3) 41-60\n"
    "4) 61-80\n"
    "5) 81-100"
)

response_percentage = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": progress_video.uri},
        {"type": "text", "text": PERCENTAGE_PROGRESS_PROMPT}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PercentProgressResponse.model_json_schema()
    }
)

print("\n--- Progress Classification Output ---")
print(json.dumps(json.loads(response_percentage.output_text), indent=2))
plot_video_progress("progress.mp4", response_percentage.output_text, label_prefix="Progress Bracket (1-5)")

#### 意味的な進捗分類


In [ ]:
class SemanticProgressResponse(BaseModel):
    digit_choice: int = Field(description="1〜4 の整数の選択肢")
    reasoning: str

SEMANTIC_PROGRESS_PROMPT = (
    "You are an expert video rater. In this task, you will be shown a video of a robot or human performing a task. "
    'The robot or human was asked to do the following task: "insert the toy screw into the top left hole on the workbench".\n'
    "How much progress did the robot or human make in completing the given task at the end of the video?\n"
    "Use the provided choices to answer the question. Possible answer choices:\n"
    "(1) No success: The final state shows no goal-relevant change for the task command.\n"
    "(2) Partial completion: The final state shows good progress but violates major requirements or multiple requirements.\n"
    "(3) Near completion: The final state is correct in region and intent but misses a single minor requirement.\n"
    "(4) Perfect completion: The final state satisfies all the requirements."
)

response_semantic = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": progress_video.uri},
        {"type": "text", "text": SEMANTIC_PROGRESS_PROMPT}
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": SemanticProgressResponse.model_json_schema()
    }
)

print("\n--- Semantic Progress Output ---")
print(json.dumps(json.loads(response_semantic.output_text), indent=2))
plot_video_progress("progress.mp4", response_semantic.output_text, label_prefix="Semantic State (1-4)")

## 発展 — 自分の写真で試す（Google Drive から読み込む）

ここからは講座オリジナルの発展編です。**自分で撮った写真**をモデルに渡して、同じように「どこ?」を聞きます。

手順は3つです。

1. [Google Drive](https://drive.google.com/) に写真をアップロードする（マイドライブ直下でOK）
2. 下のセルを実行して Colab に Drive をつなぐ（承認画面が出ます）
3. その次のセルのファイル名を自分の写真に合わせて実行する

写真が手元に無い場合は、講座リポジトリの `demo-box-wide.jpg`（デモ環境の全景写真）を
ダウンロードして、ご自身の Drive にアップロードして使ってください。

⚠️ このセルより前に、「1. セットアップ」と「2. 空間推論」冒頭のユーティリティのセルを実行しておいてください
（`client` と `load_image` / `plot_points` を使うため）。


In [ ]:
# 発展① Google Drive を Colab につなぐ（承認画面が出たら、自分のアカウントで許可）
# あわせて、グラフの日本語ラベルが文字化けしないようフォントも入れます
%pip install -q japanize-matplotlib
import japanize_matplotlib

from google.colab import drive

drive.mount('/content/drive')


Drive がつながると、左のファイルペイン（フォルダアイコン）に `drive/MyDrive/...` が現れます。
ファイル名がわからないときは、ファイルペインで写真を右クリック →「パスをコピー」で取得できます。


In [ ]:
# 発展② 自分の写真に「どこ?」を聞く — ファイル名は自分の写真に合わせて変える
PHOTO_PATH = '/content/drive/MyDrive/demo-box-wide.jpg'  # ← ここを変える

class PointItem(BaseModel):
    point: List[int] = Field(description="[y, x] format normalized to 0-1000")
    label: str = Field(description="指した場所の名前（日本語）")

class PointResponse(BaseModel):
    items: List[PointItem]

img, img_b64 = load_image(PHOTO_PATH)

prompt = (
    "この写真は自作のミニ温室2台と、その制御装置です。"
    "換気ファン、給水ボトル、マイコンボードを指してください。"
    "写っていないものは含めないでください。ラベルは日本語で。"
)

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt},
    ]}],
    generation_config={"thinking_level": "low"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": PointResponse.model_json_schema(),
    },
)

print(response.output_text)
plot_points(img, response.output_text)


## 発展その2 — ダッシュボードの画面は読めるのか

写真だけでなく、**グラフ画像**も渡せます。読ませるのは、この講座のデモ温室 A/B の**実測データ**
（1分間隔・丸1日分）を時系列データ可視化システム **RealBoard®** で表示した、**ダッシュボード画面
そのもののスクリーンショット**です。A はファン制御あり、B はファンを塞いだ対照区。

元データから計算した正解はこうです（**答え合わせができる問いを渡す**のが今日の講座全体の作法です）。

| 問い | 正解（データから計算） |
|---|---|
| A の最高温度 | 30.37℃（14:00） |
| B の最高温度 | 31.77℃（14:06） |
| A のファンが動き出した時刻 | 11:01 |

モデルは Agentic Vision で**自分でグラフを拡大しながら**読みます。
普段見ているダッシュボードの画面をそのまま渡せる、というのがポイントです。


In [ ]:
# 発展③ ダッシュボード画面を読ませる — 自分で拡大しながら読む（画像は講座リポジトリから取得）
!wget -q https://raw.githubusercontent.com/ptk-y-nakahira/iop-er2-handson/main/chart_realboard.png -O chart_realboard.png

img, img_b64 = load_image('chart_realboard.png')

prompt = (
    "この画像は、温室A/Bの1日分の実測データを表示したダッシュボード画面です。"
    "赤=A(ファン制御あり)の温度、オレンジ=B(ファン封鎖)の温度で左軸(℃)、"
    "緑=Aのファン出力で右軸(%)です。"
    "①Aの最高温度は何度で、何時ごろですか。②Aのファンが最初に動き出したのは何時ごろですか。"
    "③AとBではどちらが高温になり、差はどの時間帯に開いていますか。"
    "必要なら Python でグラフを拡大・切り出しして読んでください。"
)

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "image", "data": img_b64, "mime_type": "image/png"},
        {"type": "text", "text": prompt},
    ]}],
    generation_config={"thinking_level": "high"},
    tools=[{"type": "code_execution"}],
)

for step in response.steps:
    if step.type == "code_execution_call":
        print(f"--- モデルが書いたコード ---\n{step.arguments.code}")
    elif step.type == "code_execution_result":
        print("--- 実行結果（省略可） ---")
    elif step.type == "model_output":
        for content in step.content:
            if content.type == "text" and content.text:
                print(content.text)
            elif content.type == "image" and content.data:
                display(IPythonImage(data=base64.b64decode(content.data)))


## 発展その3 — 動画の判定（収穫機の排出）

講座で見た、あの実写映像です（USDA 撮影・パブリックドメイン・9.9秒）。
飼料ハーベスタが走りながらトラックへ刈草を吹き込んでいます。
「**排出はいつ終わったか**」を聞きます。実測の記録では、Claude（汎用）は
6秒以降の土煙を排出と見間違えて「7.4秒まで」と答え、ER 2 は「0.0〜5.0秒」と正解でした
（実際の終了は 5.2秒）。同じ質問を自分のキーで投げてみます。

⚠️ 動画のアップロードと処理に1〜2分かかります。実行は 1 リクエスト消費です。


In [ ]:
# 発展④ 収穫機の排出はいつ終わったか — 動画をアップロードして聞く
!wget -q https://raw.githubusercontent.com/ptk-y-nakahira/iop-er2-handson/main/harvest_usda.webm -O harvest_usda.webm

print("動画をアップロードしています...")
harvest_video = client.files.upload(file="harvest_usda.webm")

import time
while harvest_video.state.name == "PROCESSING":
    print(".", end="", flush=True)
    time.sleep(2)
    harvest_video = client.files.get(name=harvest_video.name)
print("\n準備完了:", harvest_video.uri)


class HarvestAnswer(BaseModel):
    end_second: float = Field(description="排出が終わった時刻（秒）。判定できなければ -1")
    reason: str = Field(description="そう判定した根拠（日本語）。6秒以降に何が見えるかにも触れる")


prompt = (
    "この動画では、収穫機がアームの先端からトラックへ刈草を吹き込んでいます。"
    "吹き込み（排出）が終わったのは何秒の時点ですか。"
    "アームの先端をよく見て判定してください。判定できない場合は -1 を返してください。"
)

response = client.interactions.create(
    model=MODEL_ID,
    input=[{"type": "user_input", "content": [
        {"type": "video", "uri": harvest_video.uri},
        {"type": "text", "text": prompt},
    ]}],
    generation_config={"thinking_level": "high"},
    response_format={
        "type": "text", "mime_type": "application/json",
        "schema": HarvestAnswer.model_json_schema(),
    },
)

print(response.output_text)


質問を変えて、何度か試してみてください（ER 2 の無料枠は 1日20リクエストです）。

- 「水をやるなら、どこに差しますか」 — 作業の場所を聞く
- 「電源タップの空きソケットを指してください」 — 公式サンプルと同じ聞き方を自分の写真で
- 「葉の状態を教えてください。必要なら拡大して構いません」 — Agentic Vision（`tools=[{"type": "code_execution"}]` を付ける）


## 発展その4 — シートの記録を読ませて、判断させる

今日の一日をつなぎます。午後の部で **ESP32 が測り、GAS がスプレッドシートにためた温湿度**を、
ここ（Colab）から読み込んで、AI に「次にどうするか」を判断させます。
デモ温室のエージェント（step3）が中でやっている判断と同じものを、自分の手元で再現する形です。

1. 下のセルの `SHEET_URL` に、**自分のスプレッドシートの URL** を貼る（ブラウザのアドレス欄からコピー）
2. 実行すると承認画面が出ます（自分のシートを読むための許可です）。
   ⚠️ アクセス範囲の画面では**チェックボックスにチェックを入れてから「続行」**してください。
   チェックせずに進むとシートが読めず、`SpreadsheetNotFound` になります

判断も **ER 2** にさせます（1日20回の枠を1回消費します）。時系列の推移から次の一手を選ぶのは、ER 2 の得意分野（progress understanding）です。


In [ ]:
# 発展その4 シートの記録を読ませて、判断させる — ESP32 → GAS → シート → Colab → AI
SHEET_URL = "ここに自分のスプレッドシートのURLを貼る"

from google.colab import auth
auth.authenticate_user()  # 承認画面が出ます（自分のシートを読むため）

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

ws = gc.open_by_url(SHEET_URL).worksheet("記録")
rows = ws.get_all_values()[-30:]  # 直近30行（約30分ぶん）
print(f"読み込み: {len(rows)} 行 / 最新: {rows[-1] if rows else 'データなし'}")

table = "\n".join(",".join(r) for r in rows)


class SheetJudgment(BaseModel):
    action: str = Field(description="次にとる操作。'fan_on' / 'fan_off' / 'none' のいずれか")
    reason: str = Field(description="そう判断した根拠（日本語）。実際の数値に触れること")


response = client.models.generate_content(
    model="gemini-robotics-er-2-preview",  # 判断も ER 2 で（1日20回の枠を1回使います）
    contents=(
        "あなたは小型温室の管理エージェントです。"
        "以下はセンサ記録シートの直近データです（各行: timestamp,temp,humid）。\n"
        + table + "\n"
        "温度の推移から次にとるべき操作を1つ選び、根拠を書いてください。"
        "目安: 30℃を超えて上昇傾向なら fan_on、28℃を下回っていれば fan_off、それ以外は none。"
    ),
    config={
        "response_mime_type": "application/json",
        "response_schema": SheetJudgment,
    },
)

judgment = response.parsed
print("判断:", judgment.action)
print("根拠:", judgment.reason)


## 次のステップ

Gemini Robotics ER 2 の詳細は[公式ドキュメント](https://ai.google.dev/gemini-api/docs/robotics-overview)を参照してください。
